# cAPTure: canonical packet preparation

This CPU-only notebook converts the completed Gate-0 audit Parquet files into the window-independent candidate `capture_packet_v1` schema. It reads the existing files from Drive, processes one scenario at a time, and writes checksum-verified prepared artifacts back to Drive. It does not download source CSVs, assign packets to windows, compute training weights, or train a model.

Run `SMOKE` first. Review the real-data transformations before approving and running `FULL_DEV`.


## 1. Mount Drive and load the project


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_ROOT = Path("/content/capture_prepare_work")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )

required_files = [
    PROJECT_ROOT / "code/python/utils/capture_prepare.py",
    PROJECT_ROOT / "code/python/tests/test_capture_prepare.py",
    PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml",
    PROJECT_ROOT / "configs/capture_experiment_v1.yaml",
    PROJECT_ROOT / "code/python/requirements-capture.txt",
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Update the repository copy first: {missing_files}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("CPU preparation environment is ready.")


## 2. Run synthetic transformation checks


In [ ]:
test_environment = dict(os.environ)
test_environment["PYTHONPATH"] = str(PROJECT_ROOT / "code/python")
subprocess.run(
    [sys.executable, "-m", "unittest", "discover",
     "-s", str(PROJECT_ROOT / "code/python/tests"),
     "-p", "test_capture_prepare.py", "-v"],
    env=test_environment,
    cwd=PROJECT_ROOT,
    check=True,
)


## 3. Configure the preparation run


In [ ]:
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display
from utils.capture_data import load_manifest, selected_scenarios, sha256_file, write_json
from utils.capture_prepare import run_capture_preparation

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
PACKET_SCHEMA_PATH = PROJECT_ROOT / "configs/capture_packet_schema_v1.yaml"
AUDIT_RUN_DIR = (
    DRIVE_ROOT / "runs" / "20260917T170658_227477Z_full_dev"
)
DECISION_AUDIT_PATH = AUDIT_RUN_DIR / "gate0_decision_audit.json"

MODE = "SMOKE"
# MODE = "FULL_DEV"
PREPARATION_SMOKE_REVIEW_PATH = None
BATCH_SIZE = 100_000

MANIFEST = load_manifest(MANIFEST_PATH)
SCENARIOS = selected_scenarios(MANIFEST, MODE)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_prepare_" + MODE.lower()
DRIVE_RUN_DIR = DRIVE_ROOT / "prepared_runs" / RUN_ID

print(f"Mode: {MODE}")
print(f"Scenarios: {SCENARIOS}")
print(f"Source audit: {AUDIT_RUN_DIR}")
print(f"Output directory: {DRIVE_RUN_DIR}")
print(f"Free local storage: {shutil.disk_usage(LOCAL_ROOT).free / 1024**3:.1f} GiB")


## 4. Prepare and persist the selected scenarios

The input audit Parquet files remain on Drive. Each output is built in temporary local storage, copied to a fresh Drive run directory, checksum-verified, and then removed locally.


In [ ]:
RESULTS = run_capture_preparation(
    manifest_path=MANIFEST_PATH,
    packet_schema_path=PACKET_SCHEMA_PATH,
    audit_run_dir=AUDIT_RUN_DIR,
    decision_audit_path=DECISION_AUDIT_PATH,
    mode=MODE,
    local_root=LOCAL_ROOT,
    drive_run_dir=DRIVE_RUN_DIR,
    smoke_review_path=(
        Path(PREPARATION_SMOKE_REVIEW_PATH)
        if PREPARATION_SMOKE_REVIEW_PATH is not None else None
    ),
    batch_size=BATCH_SIZE,
)
display(pd.DataFrame([
    {"scenario": scenario, **item}
    for scenario, item in RESULTS.items()
]))


## 5. Review the preparation reports


In [ ]:
REPORTS = {}
summary_rows = []
role_rows = []
null_rows = []

for scenario in SCENARIOS:
    report_path = DRIVE_RUN_DIR / scenario / "preparation_report.json"
    report = json.loads(report_path.read_text())
    REPORTS[scenario] = report
    summary_rows.append({
        "scenario": scenario,
        **report["counts"],
        "output_gib": report["output_size_bytes"] / 1024**3,
        "features": len(report["feature_columns"]),
        "origin_timestamp_ns": report["scenario_origin_timestamp_ns"],
        "last_timestamp_ns": report["last_packet_timestamp_ns"],
        "duration_seconds": report["duration_seconds"],
    })
    role_rows.append({"scenario": scenario, **report["destination_node_role_counts"]})
    for feature, missing in report["feature_null_counts"].items():
        null_rows.append({
            "scenario": scenario, "feature": feature, "null_values": missing,
            "null_fraction": missing / report["counts"]["packets"],
        })

print("Scenario summary")
display(pd.DataFrame(summary_rows))
print("Destination node roles")
display(pd.DataFrame(role_rows))
print("Feature null fractions")
null_frame = pd.DataFrame(null_rows)
for scenario in SCENARIOS:
    print(scenario)
    display(
        null_frame[null_frame["scenario"] == scenario]
        .drop(columns="scenario")
        .reset_index(drop=True)
    )


## 6. Record the preparation SMOKE review

Set `APPROVE_PREPARATION_SMOKE` to `True` only after reviewing Section 5. The resulting JSON authorizes `FULL_DEV` for this exact manifest, packet schema, and pair of SMOKE reports.


In [ ]:
APPROVE_PREPARATION_SMOKE = False
PREPARATION_REVIEW_NOTES = (
    "Canonical packet counts, timestamp ranges, parsed features, node roles, "
    "and null fractions were reviewed without blockers."
)

if MODE != "SMOKE":
    print("This section applies only to a SMOKE preparation run.")
elif APPROVE_PREPARATION_SMOKE:
    review = {
        "approved": True,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "packet_schema_sha256": sha256_file(PACKET_SCHEMA_PATH),
        "review_notes": PREPARATION_REVIEW_NOTES,
        "reports": {},
    }
    for scenario in SCENARIOS:
        report_path = DRIVE_RUN_DIR / scenario / "preparation_report.json"
        review["reports"][scenario] = {
            "path": str(report_path),
            "sha256": sha256_file(report_path),
        }
    review_path = DRIVE_RUN_DIR / "preparation_smoke_review.json"
    write_json(review_path, review)
    print(f"Preparation SMOKE approved: {review_path}")
else:
    print("Preparation SMOKE review remains pending. No approval was recorded.")
